# 02 — Classification Lab

Run classification on individual videos and small batches. Inspect all 8 facets, compare prompt versions.

**Cost note:** Each `classify()` call hits the Gemini Flash API. Cheap (~$0.001/call) but not free.

In [ ]:
import sys, os, json, asyncio
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if "workbench" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / "workbench" / ".env")

from app.services.gemini import classify as gemini_classify
from app.services.ontology import (
    validate_classification, format_ontology_for_prompt,
    ONTOLOGY_V1, FACET_NAMES,
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

API_KEY = os.environ["GEMINI_API_KEY"]
print(f"Loaded {len(FACET_NAMES)} facets: {FACET_NAMES}")

In [ ]:
async def classify_video(
    caption: str | None = None,
    subtitle: str | None = None,
    hashtags: list[str] | None = None,
    creator: str | None = None,
    music: str | None = None,
) -> dict:
    """Classify a video and return structured result with raw, tier1, tier2, confidence."""
    result = await gemini_classify(
        api_key=API_KEY,
        caption=caption,
        subtitle=subtitle,
        hashtags=hashtags,
        creator_username=creator,
        music_name=music,
    )
    if not result.success:
        return {"success": False, "error": result.error}
    
    validated = validate_classification(result.raw_classification or {})
    return {
        "success": True,
        "raw": result.raw_classification,
        "tier1": validated.tier1,
        "tier2": validated.tier2,
        "confidence": validated.confidence,
    }

In [ ]:
# Classify a single video
result = await classify_video(
    caption="this pasta recipe changed my life fr fr",
    hashtags=["#pasta", "#cooking", "#recipe", "#foodtiktok"],
    creator="@homecookemily",
    music="original sound - homecookemily",
)

if result["success"]:
    print("=== Tier-1 Labels ===")
    for facet in FACET_NAMES:
        label = result["tier1"].get(facet, "—")
        conf = result["confidence"].get(facet, 0)
        print(f"  {facet:25s} {label:20s} ({conf:.0%})")
    print("\n=== Tier-2 Micro-Labels ===")
    for facet, labels in result["tier2"].items():
        if labels:
            print(f"  {facet:25s} {', '.join(labels)}")
else:
    print(f"Classification failed: {result['error']}")

In [ ]:
# Batch classify from sample-videos.json
samples_path = REPO_ROOT / "workbench" / "data" / "sample-videos.json"
samples = json.loads(samples_path.read_text())

if not samples:
    print("sample-videos.json is empty. Generate test data first:")
    print("  python workbench/scripts/generate_test_data.py 'diverse tiktok videos' --count 10")
else:
    results = []
    for item in samples:
        r = await classify_video(
            caption=item.get("caption"),
            subtitle=item.get("subtitle"),
            hashtags=item.get("hashtags"),
            creator=item.get("creator"),
            music=item.get("music"),
        )
        r["id"] = item.get("id", "unknown")
        results.append(r)
    
    # Build DataFrame of tier1 results
    rows = [{"id": r["id"], **r.get("tier1", {})} for r in results if r["success"]]
    if rows:
        df = pd.DataFrame(rows)
        print(f"Classified {len(rows)}/{len(samples)} successfully")
        df

In [ ]:
# Per-facet confidence distribution
if samples and any(r["success"] for r in results):
    conf_data = []
    for r in results:
        if r["success"]:
            for facet, score in r["confidence"].items():
                conf_data.append({"facet": facet, "confidence": score})
    
    conf_df = pd.DataFrame(conf_data)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=conf_df, x="facet", y="confidence", ax=ax, color="#E6E4DE")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_title("Confidence Distribution by Facet")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

In [ ]:
# Confusion analysis against golden set
golden_path = REPO_ROOT / "workbench" / "data" / "golden-set.json"
golden = json.loads(golden_path.read_text())

if not golden:
    print("golden-set.json is empty. Add hand-labeled items to evaluate accuracy.")
else:
    golden_results = []
    for item in golden:
        r = await classify_video(
            caption=item.get("caption"),
            subtitle=item.get("subtitle"),
            hashtags=item.get("hashtags"),
            creator=item.get("creator"),
            music=item.get("music"),
        )
        expected = item.get("expected", {})
        for facet, expected_label in expected.items():
            predicted = r.get("tier1", {}).get(facet, "—")
            golden_results.append({
                "id": item["id"], "facet": facet,
                "expected": expected_label, "predicted": predicted,
                "correct": expected_label == predicted,
            })
    
    gdf = pd.DataFrame(golden_results)
    accuracy = gdf.groupby("facet")["correct"].mean()
    print("Per-Facet Accuracy:")
    for facet, acc in accuracy.items():
        print(f"  {facet:25s} {acc:.0%}")
    print(f"\nOverall: {gdf['correct'].mean():.0%}")

## Prompt Comparison

Save alternative prompt functions in `workbench/evals/prompts/` as Python files, then import and compare.

In [ ]:
# Side-by-side prompt comparison
# To use: create workbench/evals/prompts/v2_prompt.py with an alternative classify function
# Then uncomment and adapt:
#
# sys.path.insert(0, str(REPO_ROOT / "workbench" / "evals" / "prompts"))
# from v2_prompt import classify as classify_v2
#
# test_video = {"caption": "...", "hashtags": ["..."]}
# result_v1 = await classify_video(**test_video)
# result_v2 = await classify_v2(**test_video)  # your alternative
#
# for facet in FACET_NAMES:
#     v1 = result_v1["tier1"].get(facet, "—")
#     v2 = result_v2.get(facet, "—")
#     match = "✓" if v1 == v2 else "✗"
#     print(f"  {match} {facet:25s} v1={v1:20s} v2={v2}")

print("Prompt comparison placeholder — see instructions above.")